# Read and Explore

This tutorial shows how to load data from an `.xlsx` file into Julia using [XLSX.jl](https://github.com/felipenoris/XLSX.jl) and the [Tables.jl](https://github.com/JuliaData/Tables.jl) interface. No Excel installation is needed — XLSX.jl reads the Office Open XML format directly.

In [ ]:
using XLSX

## Write a Sample File

Create a small workbook to read back in the cells below.

In [ ]:
path = tempname() * ".xlsx"
XLSX.openxlsx(path; mode="w") do xf
    ws = xf[1]
    ws["A1"] = "name";  ws["B1"] = "score";  ws["C1"] = "grade"
    ws["A2"] = "Alice"; ws["B2"] = 95;       ws["C2"] = "A"
    ws["A3"] = "Bob";   ws["B3"] = 82;       ws["C3"] = "B"
    ws["A4"] = "Carol"; ws["B4"] = 74;       ws["C4"] = "C"
    XLSX.addsheet!(xf, "Scores2")
    ws2 = xf["Scores2"]
    ws2["A1"] = "name"; ws2["B1"] = "score"
    ws2["A2"] = "Dave"; ws2["B2"] = 88
end
println("Written: ", path)

## Inspect the Workbook

Open the workbook and list its sheets.

In [ ]:
XLSX.openxlsx(path) do xf
    println("Sheets: ", XLSX.sheetnames(xf))
    ws = xf[1]
    println("Dimensions: ", XLSX.get_dimension(ws))
    println("A1:C4 = ", ws["A1:C4"])  # Matrix{Any}
end

## Read with Tables.jl Interface

`XLSX.eachtablerow` returns a Tables.jl-compatible row iterator. The first row is treated as the header — columns are accessible by name.

In [ ]:
XLSX.openxlsx(path) do xf
    for row in XLSX.eachtablerow(xf[1])
        println(row[:name], "  score=", row[:score], "  grade=", row[:grade])
    end
end

## Read All Sheets into a Dict

Collect every sheet's rows into a `Dict` of `Vector{NamedTuple}` — no DataFrames needed.

In [ ]:
XLSX.openxlsx(path) do xf
    for name in XLSX.sheetnames(xf)
        rows = [NamedTuple(row) for row in XLSX.eachtablerow(xf[name])]
        println(name, ": ", length(rows), " rows — ", rows)
    end
end

## Summary
- `XLSX.openxlsx(path) do xf ... end` opens a workbook safely — no manual close needed.
- `XLSX.sheetnames(xf)` lists available sheets; `xf[1]` or `xf["Name"]` accesses one.
- `ws["A1:C4"]` returns a `Matrix{Any}` for any rectangular range.
- `XLSX.eachtablerow(ws)` is a Tables.jl row iterator: first row is the header, columns accessible by name.
- Collect rows into `NamedTuple` vectors for lightweight tabular data without DataFrames.